In [3]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
import regex as re
print(re.findall(PAT, "some text that I'll pretokenize. H"))
a = re.finditer(PAT, "some")
for m in a:
    print(m.start())

['some', ' text', ' that', ' I', "'ll", ' pretokenize', '.', ' H']
0


In [ ]:
import os
print(os.path.getsize("../data/TinyStoriesV2-GPT4-train.txt") / (1024**3), "GB")
print(os.path.getsize("../data/owt_train.txt") / (1024**3), "GB")
# Output:
# 2.074756810441613 GB
# 11.101841050200164 GB

2.074756810441613 GB
11.101841050200164 GB


In [ ]:
# if __name__ == "__main__":
import train_bpe_notebook
# from multiprocessing import freeze_support
# freeze_support()
print(os.listdir('..'))
tiny_stories = train_bpe_notebook.TrainBPE('../data/TinyStoriesV2-GPT4-train.txt', 10000, ["<|endoftext|>"])
# Took 4m 23.7s

['.git', '.gitignore', '.pytest_cache', '.venv', 'CHANGELOG.md', 'cs336_basics', 'cs336_spring2025_assignment1_basics.pdf', 'data', 'LICENSE', 'make_submission.sh', 'pyproject.toml', 'README.md', 'tests', 'uv.lock']
test:  <Future at 0x2afff299e50 state=finished returned dict>
test:  <Future at 0x2afff29a0d0 state=finished returned dict>
test:  <Future at 0x2afff25ed70 state=finished returned dict>
test:  <Future at 0x2afff25eb10 state=finished returned dict>
test:  <Future at 0x2afff2779b0 state=finished returned dict>
test:  <Future at 0x2afff24d9d0 state=finished returned dict>
test:  <Future at 0x2afff24d6a0 state=finished returned dict>
test:  <Future at 0x2afff256c50 state=finished returned dict>
test:  <Future at 0x2afff256b50 state=finished returned dict>
test:  <Future at 0x2afff23ff20 state=finished returned dict>
test:  <Future at 0x2afff23fd40 state=finished returned dict>
test:  <Future at 0x2afff249e10 state=finished returned dict>
test:  <Future at 0x2afff24a0b0 state=fi

In [6]:
import json
fout = open("tiny_stories_vocab.json", 'w')
json.dump({key: str(value) for key, value in tiny_stories[0].items()}, fout)
fout.close()
fout = open("tiny_stories_merge.json", 'w')
json.dump([(str(i[0]), str(i[1])) for i in tiny_stories[1]], fout)
fout.close()

In [ ]:
# if __name__ == "__main__":
import train_bpe_notebook
# from multiprocessing import freeze_support
# freeze_support()
print(os.listdir('..'))
owt = train_bpe_notebook.TrainBPE('../data/owt_train.txt', 32000, ["<|endoftext|>"])
# Took 229m 0.9s

['.git', '.gitignore', '.pytest_cache', '.venv', 'CHANGELOG.md', 'cs336_basics', 'cs336_spring2025_assignment1_basics.pdf', 'data', 'LICENSE', 'make_submission.sh', 'pyproject.toml', 'README.md', 'tests', 'uv.lock']
test:  <Future at 0x2afff213550 state=finished returned dict>
test:  <Future at 0x2afff2132d0 state=finished returned dict>
test:  <Future at 0x2afff2137d0 state=finished returned dict>
test:  <Future at 0x2afff2136d0 state=finished returned dict>
test:  <Future at 0x2afff2138d0 state=finished returned dict>
test:  <Future at 0x2afff213950 state=finished returned dict>
test:  <Future at 0x2afff2139d0 state=finished returned dict>
test:  <Future at 0x2afff213a50 state=finished returned dict>
test:  <Future at 0x2afff213ad0 state=finished returned dict>
test:  <Future at 0x2afff213bd0 state=finished returned dict>
test:  <Future at 0x2afff213c50 state=finished returned dict>
test:  <Future at 0x2afff213cd0 state=finished returned dict>
test:  <Future at 0x2afff213d50 state=fi

In [8]:
import json
fout = open("owt_vocab.json", 'w')
json.dump({key: str(value) for key, value in owt[0].items()}, fout)
fout.close()
fout = open("owt_merge.json", 'w')
json.dump([(str(i[0]), str(i[1])) for i in owt[1]], fout)
fout.close()

In [19]:
"""
adapter.py run commands:
$env:PYTHONUTF8 = "1"
uv run pytest tests/test_tokenizer.py
"""
from __future__ import annotations
from collections.abc import Iterable, Iterator
import regex as re
PAT = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

class Tokenizer:
    def __init__(self, 
                vocab: dict[int, bytes],
                merges: list[tuple[bytes, bytes]],
                special_tokens: list[str] | None = None):
        self.vocab = vocab
        self.invert_vocab = {v: k for k, v in vocab.items()}
        self.merges = merges
        if special_tokens:
            self.special_tokens = special_tokens
        else:
            self.special_tokens = []
        
        self.merge_order = {m: i for i, m in enumerate(self.merges)}
        self.special_order = sorted(self.special_tokens, key=len, reverse=True)
        self.special_hash = set(self.special_order)
    
    @classmethod
    def from_files(cls, vocab_filepath: str, merges_filepath: str, special_tokens: list[str] | None = None):
        import json

        with open(vocab_filepath, encoding="utf-8") as fin:
            vocab = json.load(fin)
        
        with open(merges_filepath, encoding="utf-8") as fin:
            merges = json.load(fin)

        return cls(vocab=vocab, merges=merges, special_tokens=special_tokens)
    
    def encode(self, text: str) -> list[int]:
        return list(self.encode_iterable([text]))

    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        for cur in iterable:
            text = cur.replace("\r\n", "\n").replace("\r", "\n")
            if self.special_order:
                # Remove all the special tokens before pretokenization
                splitter = re.compile("(" + "|".join(re.escape(token) for token in self.special_order) + ")")
                split_text = splitter.split(text)
            else:
                split_text = [text]

            for segment in split_text:
                if segment in self.special_hash:
                    yield self.invert_vocab[segment.encode("utf-8")]
                    continue
                for i in PAT.finditer(segment):
                    pretoken = i.group(0)

                    btext = [bytes([b]) for b in pretoken.encode("utf-8")]
                    # Either set up priority queue or quadratic solution

                    while True:
                        min_count = len(self.merge_order)
                        merge = None
                        for j in range(len(btext) - 1):
                            cur_merge = (btext[j], btext[j + 1])
                            cur_count = self.merge_order.get(cur_merge, min_count)
                            if min_count > cur_count:
                                min_count = cur_count
                                merge = cur_merge
                        if min_count == len(self.merge_order):
                            break

                        btemp = []
                        j = 0
                        while j < len(btext):
                            if (j < len(btext) - 1) and ((btext[j], btext[j + 1]) == merge):
                                btemp.append(btext[j] + btext[j + 1])
                                j += 2
                            else:
                                btemp.append(btext[j])
                                j += 1
                        btext = btemp
                
                    # newbtext = []
                    # for merge in self.merges:
                    #     for i in range(len(btext) - 1):
                    #         if merge == (btext[i], btext[i + 1]):
                    #             flag = True
                    #             newbtext.append(btext[i] + btext[i + 1])
                    #             i += 1
                    #         else:
                    #             flag = False
                    #             newbtext.append(btext[i])
                    #     if not flag:
                    #         newbtext.append(btext[-1])
                    #     btext = newbtext
                    #     newbtext = []
                    
                    for j in btext:
                        yield self.invert_vocab[j]
        
    def decode(self, ids: list[int]) -> str:
        cur = bytearray()
        for id in ids:
            cur.extend(self.vocab[id])
        
        return cur.decode("utf-8", errors="replace")

In [ ]:
tinystories_tokenizer = Tokenizer(tiny_stories[0], tiny_stories[1], ["<|endoftext|>"])
ts_sample_encode = [len(tiny_stories[0][i]) for i in tinystories_tokenizer.encode_iterable(open('../data/tinystories_sample.txt', encoding="utf-8"))]
print(sum(ts_sample_encode) / len(ts_sample_encode))

# Output 4.105615448406587

4.105615448406587


In [ ]:
owt_tokenizer = Tokenizer(owt[0], owt[1], ["<|endoftext|>"])
owt_sample_encode = [len(owt[0][i]) for i in owt_tokenizer.encode_iterable(open('../data/owt_sample.txt', encoding="utf-8"))]
print(sum(owt_sample_encode) / len(owt_sample_encode))

# Output 4.5645572488810835

4.5645572488810835


In [ ]:
# Much smaller than above (3.389524610854018 < both 4.105615448406587 and 4.5645572488810835, respectively as above) - in this case, since we're merging byte pairs,
# Bigger number = better/more efficient merging. Makes sense that this is less efficient since we didn't train and test on the same dataset in this case.
from time import time
start = time()
owt_ts_tokenizer_sample_encode = []
for i in tinystories_tokenizer.encode_iterable(open('../data/owt_sample.txt', encoding="utf-8")):
    owt_ts_tokenizer_sample_encode.append(len(tiny_stories[0][i]))
print(sum(owt_ts_tokenizer_sample_encode) / len(owt_ts_tokenizer_sample_encode))
throughput = sum(owt_ts_tokenizer_sample_encode) / (time() - start)
print(throughput)
# Output:
# 3.389524610854018
# 519138.1143382093

3.389524610854018
519138.1143382093


In [ ]:
# Estimated Pile dataset tokenization time in seconds (1589172.471090279 seconds = 18.3931999 days)
print((825 * 1000000000) / throughput)
# Output 1589172.471090279

1589172.471090279


In [26]:
import numpy as np

In [29]:
ts_enc_train = np.fromiter(tinystories_tokenizer.encode_iterable(open('../data/TinyStoriesV2-GPT4-train.txt', encoding="utf-8")), dtype = np.uint16, count = -1)

In [30]:
ts_enc_valid = np.fromiter(tinystories_tokenizer.encode_iterable(open('../data/TinyStoriesV2-GPT4-valid.txt', encoding="utf-8")), dtype = np.uint16, count = -1)

In [1]:
owt_enc_train = np.fromiter(owt_tokenizer.encode_iterable(open('../data/owt_train.txt', encoding="utf-8")), dtype = np.uint16, count = -1)

NameError: name 'np' is not defined

In [ ]:
owt_enc_valid = np.fromiter(owt_tokenizer.encode_iterable(open('../data/owt_valid.txt', encoding="utf-8")), dtype = np.uint16, count = -1)